In [ ]:

import pandas as pd        
import numpy as np  

from sklearn.model_selection import train_test_split   
from sklearn.compose import ColumnTransformer          
from sklearn.preprocessing import OneHotEncoder        
from sklearn.pipeline import Pipeline                  
from sklearn.linear_model import LogisticRegression    



from sklearn.metrics import (
    confusion_matrix,          
    classification_report,     
    accuracy_score,            
)



import matplotlib.pyplot as plt  

print("Librerias cargadas correctamente.")

In [ ]:
# Cargar el archivo CSV en un DataFrame (tabla de datos)
df = pd.read_csv("abandono_clientes.csv")

# Mostrar las primeras 5 filas para verificar que se cargo correctamente
print("Primeras 5 filas del dataset:")
df.head()

In [ ]:
# Informacion general del dataset
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
print()

# Tipo de dato de cada columna
print("Tipos de datos:")
for col, dtype in df.dtypes.items():
    print(f"  {col:20s} -> {dtype}")

print()

# Verificar valores faltantes
nulos = df.isna().sum()
if nulos.sum() == 0:
    print("No hay valores faltantes en el dataset.")
else:
    print(f"ATENCION: hay {nulos.sum()} valores nulos.")
    print(nulos[nulos > 0])

In [ ]:
# Estadisticas descriptivas de las variables numericas
df.describe().round(2)

In [ ]:
# ---------------------------------------------------------------
# DISTRIBUCION DE LA VARIABLE OBJETIVO
# ---------------------------------------------------------------
# Esto es MUY importante en clasificacion.
# Si el 99% de clientes no abandona, un modelo que siempre diga "no abandona"
# tendria 99% de accuracy pero seria completamente inutil.
# Necesitamos ver que tan balanceadas estan las clases.

conteo = df["abandono"].value_counts()
porcentaje = df["abandono"].value_counts(normalize=True) * 100

print("Distribucion de la variable objetivo (abandono):")
print(f"  Permanece (0): {conteo[0]} clientes ({porcentaje[0]:.1f}%)")
print(f"  Abandona  (1): {conteo[1]} clientes ({porcentaje[1]:.1f}%)")
print()
print("Las clases estan moderadamente desbalanceadas.")
print("Hay mas clientes que permanecen que clientes que abandonan.")
print("Esto es lo tipico en la realidad: la mayoria de clientes NO se van.")

In [ ]:
# Comparar el promedio de cada variable numerica segun si el cliente abandono o no
# groupby("abandono") separa los datos en dos grupos: 0 (permanece) y 1 (abandona)
# .mean() calcula el promedio de cada variable en cada grupo

comparacion = df.groupby("abandono")[["antiguedad_meses", "factura_mensual",
                                       "llamadas_soporte", "satisfaccion"]].mean().round(2)

# Renombrar el indice para que sea mas legible
comparacion.index = ["Permanece (0)", "Abandona (1)"]

print("Promedio de cada variable segun abandono:")
print()
comparacion

In [ ]:
# Analisis del tipo de contrato vs abandono
# pd.crosstab crea una tabla cruzada: filas = contrato, columnas = abandono
# normalize="index" muestra porcentajes por fila (cada tipo de contrato suma 100%)

tabla_contrato = pd.crosstab(
    df["contrato"],
    df["abandono"],
    normalize="index"  # Porcentajes por fila
).round(3) * 100

tabla_contrato.columns = ["Permanece (%)", "Abandona (%)"]

print("Porcentaje de abandono por tipo de contrato:")
print()
tabla_contrato

In [ ]:
# --- Paso 1: Separar X (variables predictoras) e y (variable objetivo) ---
X = df[["antiguedad_meses", "factura_mensual", "llamadas_soporte",
        "contrato", "satisfaccion"]]
y = df["abandono"]

# --- Paso 2: Dividir en entrenamiento (80%) y prueba (20%) ---
# stratify=y mantiene la misma proporcion de abandono en ambos conjuntos.
# Ejemplo: si hay 35% de abandono en el total, habra ~35% en entrenamiento y ~35% en prueba.
# Esto es importante cuando las clases estan desbalanceadas.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% para prueba
    random_state=42,     # Semilla para reproducibilidad
    stratify=y           # Mantener proporcion de clases
)

print(f"Datos de entrenamiento: {X_train.shape[0]} clientes")
print(f"Datos de prueba:        {X_test.shape[0]} clientes")
print()
print(f"Proporcion de abandono en entrenamiento: {y_train.mean()*100:.1f}%")
print(f"Proporcion de abandono en prueba:        {y_test.mean()*100:.1f}%")
print()
print("Las proporciones son similares gracias al parametro stratify=y.")

In [ ]:
# --- Definir columnas por tipo ---
numeric_features = ["antiguedad_meses", "factura_mensual",
                    "llamadas_soporte", "satisfaccion"]
categorical_features = ["contrato"]

# --- Preprocesamiento ---
# Las columnas numericas se dejan tal cual (passthrough).
# La columna categorica (contrato) se convierte a columnas binarias con OneHotEncoder.
# Ejemplo: contrato="Mensual" -> contrato_Mensual=1, contrato_Anual=0, contrato_Dos_anios=0
preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

# --- Pipeline: preprocesamiento + modelo ---
# El pipeline garantiza que los datos siempre se preprocesen antes de llegar al modelo.
# max_iter=1000 le da al modelo suficientes iteraciones para converger (encontrar la solucion).
pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000)),
])

# --- Entrenar el modelo ---
# .fit() hace que el modelo aprenda los patrones de los datos de entrenamiento.
# Internamente calcula los pesos (coeficientes) que mejor separan a los clientes
# que abandonan de los que permanecen.
pipe.fit(X_train, y_train)

print("Modelo entrenado correctamente.")

In [ ]:
# Generar predicciones con los datos de prueba
y_pred = pipe.predict(X_test)

# Mostrar las primeras 10 predicciones vs valores reales
comparacion_pred = pd.DataFrame({
    "Real":      y_test.values[:10],
    "Prediccion": y_pred[:10],
    "Correcto":  ["Si" if r == p else "NO" for r, p in zip(y_test.values[:10], y_pred[:10])]
})

print("Primeras 10 predicciones vs valores reales:")
print()
comparacion_pred

In [ ]:
# Calcular el accuracy
acc = accuracy_score(y_test, y_pred)

print(f"Accuracy (exactitud): {acc:.4f} ({acc*100:.1f}%)")
print()
print(f"Esto significa que el modelo acerto {int(acc * len(y_test))} de {len(y_test)} predicciones.")
print()
print("Pero... cuantos clientes que realmente abandonaron logro detectar?")
print("Para responder esa pregunta, necesitamos la MATRIZ DE CONFUSION.")

In [ ]:
# Calcular la matriz de confusion
cm = confusion_matrix(y_test, y_pred)

# Extraer los 4 valores de la matriz
VN, FP, FN, VP = cm.ravel()

print("=" * 50)
print("          MATRIZ DE CONFUSION")
print("=" * 50)
print()
print(f"                  Prediccion del modelo")
print(f"                  Permanece(0)  Abandona(1)")
print(f"  Real Perm.(0)       {VN:3d}           {FP:3d}")
print(f"  Real Aban.(1)       {FN:3d}           {VP:3d}")
print()
print("=" * 50)
print()
print("Lectura de la matriz:")
print(f"  Verdaderos Negativos (VN): {VN} -- Dijo 'permanece' y acerto")
print(f"  Falsos Positivos     (FP): {FP} -- Dijo 'abandona' pero no se fue (falsa alarma)")
print(f"  Falsos Negativos     (FN): {FN} -- Dijo 'permanece' pero SI se fue (NO lo detecto)")
print(f"  Verdaderos Positivos (VP): {VP} -- Dijo 'abandona' y acerto")

In [ ]:
# --- Visualizacion grafica de la matriz de confusion ---
fig, ax = plt.subplots(figsize=(6, 5))

# Dibujar la matriz como un mapa de calor
im = ax.imshow(cm, cmap="Blues")

# Etiquetas de los ejes
labels = ["Permanece (0)", "Abandona (1)"]
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)
ax.set_xlabel("Prediccion del modelo")
ax.set_ylabel("Valor real")
ax.set_title("Matriz de Confusion")

# Colocar los numeros dentro de cada celda
nombres = [[f"VN\n{VN}", f"FP\n{FP}"],
           [f"FN\n{FN}", f"VP\n{VP}"]]
for i in range(2):
    for j in range(2):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, nombres[i][j], ha="center", va="center",
                fontsize=14, fontweight="bold", color=color)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# --- Calcular Precision y Recall manualmente para entender la formula ---
precision_manual = VP / (VP + FP) if (VP + FP) > 0 else 0
recall_manual = VP / (VP + FN) if (VP + FN) > 0 else 0
f1_manual = 2 * (precision_manual * recall_manual) / (precision_manual + recall_manual) if (precision_manual + recall_manual) > 0 else 0

print("CALCULO MANUAL (para la clase 'abandona'):")
print(f"  Precision = VP / (VP + FP) = {VP} / ({VP} + {FP}) = {precision_manual:.4f} ({precision_manual*100:.1f}%)")
print(f"  Recall    = VP / (VP + FN) = {VP} / ({VP} + {FN}) = {recall_manual:.4f} ({recall_manual*100:.1f}%)")
print(f"  F1-Score  = 2 * (P * R) / (P + R) = {f1_manual:.4f} ({f1_manual*100:.1f}%)")
print()
print("Ahora veamos el reporte completo generado por scikit-learn:")

In [ ]:
# --- Reporte completo de clasificacion ---
# classification_report genera un reporte con Precision, Recall y F1
# para AMBAS clases (0 y 1) y promedios generales.
print("=" * 55)
print("       REPORTE DE CLASIFICACION")
print("=" * 55)
print()
print(classification_report(
    y_test, y_pred,
    target_names=["Permanece (0)", "Abandona (1)"]
))
print("=" * 55)
print()
print("COMO LEER ESTE REPORTE:")
print("  - Cada fila es una clase (Permanece o Abandona).")
print("  - precision: de los que el modelo dijo que son de esta clase, cuantos acerto.")
print("  - recall: de los que realmente son de esta clase, cuantos detecto.")
print("  - f1-score: promedio armonico entre precision y recall.")
print("  - support: cantidad de muestras reales de cada clase en el set de prueba.")

In [ ]:
# --- Simulacion del costo economico de los errores ---
# Supongamos estos costos estimados:
costo_fp = 20     # Costo de contactar a un cliente que no se iba ($20: llamada + descuento)
costo_fn = 500    # Costo de perder a un cliente ($500: ingresos perdidos)

# Calcular el costo total de los errores del modelo
costo_total_fp = FP * costo_fp
costo_total_fn = FN * costo_fn
costo_total = costo_total_fp + costo_total_fn

# Calcular el costo si NO usaramos modelo (todos los que abandonan se pierden)
total_abandonos_reales = VP + FN  # Total de clientes que realmente abandonaron
costo_sin_modelo = total_abandonos_reales * costo_fn

# Ahorro gracias al modelo
ahorro = costo_sin_modelo - costo_total

print("=" * 55)
print("    ANALISIS DE COSTO DE LOS ERRORES")
print("=" * 55)
print()
print(f"  Costo por Falso Positivo (falsa alarma): ${costo_fp}")
print(f"  Costo por Falso Negativo (no detectado): ${costo_fn}")
print()
print(f"  Falsos Positivos: {FP} x ${costo_fp} = ${costo_total_fp:,}")
print(f"  Falsos Negativos: {FN} x ${costo_fn} = ${costo_total_fn:,}")
print(f"  Costo total de errores: ${costo_total:,}")
print()
print(f"  Si NO usaramos modelo (perder a todos): ${costo_sin_modelo:,}")
print(f"  Ahorro gracias al modelo: ${ahorro:,}")
print()
print("  Los Falsos Negativos (clientes no detectados) representan")
print(f"  el {costo_total_fn/costo_total*100:.0f}% del costo total de errores.")
print("  Esto confirma que en este problema, el Recall es critico.")